# Step 2 — Risk Enrichment (FDA FAERS + NIH Reporter)

**TrialWatch | APAN5400 Data Engineering | Group 8 | Spring 2026**

## What this notebook does
This notebook implements **Step 2** of the TrialWatch ETL pipeline — enriching the
132,185 ACT trials from Step 1 with two additional federal data sources:

1. **FDA FAERS** — 65.7M adverse event reports matched to trial drug ingredients
2. **NIH Reporter API** — Public grant funding linked to noncompliant sponsor organizations

### Processing steps:
1. Load compliance output from Step 1 (`compliancemetrics.csv`, `trialsclean.csv`)
2. Extract and process all FAERS DRUG files with PySpark (65.7M rows)
3. Fetch drug/intervention names for each noncompliant trial via ClinicalTrials.gov API v2
4. Match trial drug names to FAERS active ingredients (exact + partial match)
5. Assign danger scores (0–100) and tiers (CRITICAL/HIGH/MODERATE/LOW/NO DATA)
6. Query NIH Reporter API for public funding per sponsor organization
7. Deduplicate AE counts and funding to avoid double-counting in rollups
8. Output: `risk_enrichment.csv` → loaded into MongoDB `risk_enrichment` collection

## Why PySpark for FAERS?
FAERS has 65.7M rows across ~100 quarterly ZIP files. pandas would run out of memory.
PySpark processes each file incrementally and aggregates AE counts per ingredient.


In [ ]:
# Mount Google Drive and locate Step 1 output files
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Verify Step 1 outputs exist before proceeding
!find '/content/drive' -name 'compliancemetrics.csv' -type f 2>/dev/null | head -5
!find '/content/drive' -name 'trialsclean*' -type f 2>/dev/null | head -5


## Cell 2 — Load Step 1 Outputs
Loads `compliancemetrics.csv` and `trialsclean.csv` from Google Drive.
Filters compliance data to only noncompliant trials (LATE + MISSING) — these are
the 92,757 trials we need to enrich with danger scores and funding data.

In [ ]:
import pandas as pd
import numpy as np
import glob
import os
import zipfile
import requests
import time
import re

# ── FILE PATHS ────────────────────────────────────────────────────────────────
# BASE_SPARK: folder containing Step 1 Spark outputs
# BASE_OLD: folder containing FAERS ZIP files
BASE_SPARK = '/content/drive/.shortcut-targets-by-id/1RHn-liAsgGH31DsJuv0hJrog3w2if1zt/Data/step 1 - clinical trial w spark/'
BASE_OLD   = '/content/drive/.shortcut-targets-by-id/1_XE0-4_DTuI64Jsm6Ngp3XmHmjt6H8QI/Clinical trial '

# ── LOAD COMPLIANCE DATA ──────────────────────────────────────────────────────
# compliancemetrics.csv output from Step 1 (PySpark)
compliance = pd.read_csv(BASE_SPARK + 'compliancemetrics.csv')

# Filter to only LATE and MISSING trials — these are the noncompliant ones we enrich
# COMPLIANT and NOT_DUE_YET trials don't need danger scores
noncompliant = compliance[compliance['compliancestatusv2'].isin(['LATE', 'MISSING'])]
print(f"Total non-compliant trials: {len(noncompliant)}")
print(noncompliant['compliancestatusv2'].value_counts())

# ── LOAD TRIAL METADATA ───────────────────────────────────────────────────────
# trialsclean.csv contains org_name, phases etc. needed for the merge
trials = pd.read_csv(BASE_SPARK + 'trialsclean.csv')
print(f"\nTrials loaded: {trials.shape}")
print(f"Columns: {list(trials.columns)}")


## Cell 3 — Extract & Process FDA FAERS with PySpark
FDA FAERS (Adverse Event Reporting System) data arrives as quarterly ZIP files,
each containing a DRUG.txt file with pipe-delimited adverse event records.

This cell:
- Extracts all FAERS ZIPs to local Colab storage
- Processes each DRUG.txt file with PySpark
- Filters to **Primary Suspect (PS)** drugs only (the drug most likely causing the AE)
- Counts unique adverse event reports per active ingredient using `countDistinct(primaryid)`
- Result: a DataFrame mapping drug ingredient → total unique AE reports

In [ ]:
# ── EXTRACT ALL FAERS ZIP FILES ───────────────────────────────────────────────
# FAERS is distributed as quarterly ZIPs from FDA website
# We extract all of them to local Colab disk for faster Spark processing
FAERS_DRIVE = BASE_OLD + '/FAERS/'
FAERS_LOCAL = '/content/faers_extracted/'
os.makedirs(FAERS_LOCAL, exist_ok=True)

zip_files = sorted(glob.glob(FAERS_DRIVE + '*.zip'))
print(f"Found {len(zip_files)} ZIP files")

for zf in zip_files:
    print(f"Unzipping: {os.path.basename(zf)}")
    with zipfile.ZipFile(zf, 'r') as z:
        z.extractall(FAERS_LOCAL)

# Find all extracted DRUG files (one per quarterly ZIP)
drug_files = sorted(glob.glob(FAERS_LOCAL + '**/DRUG*.txt', recursive=True))
print(f"Total DRUG files found: {len(drug_files)}")


In [ ]:
!pip install pyspark -q

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, upper, trim, countDistinct
from collections import Counter

# ── SPARK SESSION FOR FAERS PROCESSING ───────────────────────────────────────
# 8g driver memory needed to handle aggregation of 65.7M FAERS rows
spark = SparkSession.builder \
    .appName("TrialWatch-FAERS") \
    .config("spark.driver.memory", "8g") \
    .getOrCreate()

print(f"Processing {len(drug_files)} DRUG files with Spark...")

# Counter accumulates AE counts across all quarterly files
ae_counter  = Counter()
total_rows  = 0
total_ps    = 0

for f in drug_files:
    try:
        # FAERS DRUG files are $ delimited (not comma or pipe)
        df = spark.read.csv(f, header=True, sep="$", inferSchema=False)
        rows = df.count()
        total_rows += rows

        # Filter to PRIMARY SUSPECT (role_cod == 'PS') only
        # PS = the drug most likely responsible for the adverse event
        # Excludes concomitant and interacting drugs to avoid overcounting
        ps = df.filter(col("role_cod") == "PS")
        total_ps += ps.count()

        # Count DISTINCT primaryids (unique AE reports) per active ingredient
        # prod_ai = active ingredient field in FAERS DRUG table
        counts = ps.withColumn("prod_ai_clean", upper(trim(col("prod_ai")))) \
            .groupBy("prod_ai_clean") \
            .agg(countDistinct("primaryid").alias("ae_count")) \
            .toPandas()

        # Accumulate counts into Counter (summed across quarters)
        for _, row in counts.iterrows():
            ae_counter[row['prod_ai_clean']] += row['ae_count']

        print(f"  {os.path.basename(f)}: {rows} rows")
    except Exception as e:
        print(f"  SKIPPED {os.path.basename(f)}: {e}")

# Convert final Counter to a sorted DataFrame
ae_counts = pd.DataFrame(ae_counter.most_common(), columns=['active_ingredient', 'ae_count'])

print(f"\nTotal FAERS rows processed: {total_rows:,}")
print(f"Unique active ingredients: {len(ae_counts):,}")
print("\nTop 15 by AE count:")
print(ae_counts.head(15).to_string())

spark.stop()  # Free memory before pandas-heavy steps below


## Cell 5 — Fetch Drug Names from ClinicalTrials.gov API
To match trials to FAERS, we need to know what drug each trial tested.
This data isn't in the AACT flat files — we fetch it via the ClinicalTrials.gov API v2.

Batches 100 NCT IDs per request. 0.3s delay between requests to respect rate limits.
Result: a dict mapping `nct_id → intervention names string`.

In [ ]:
def fetch_interventions_batch(nct_ids):
    """
    Fetch drug/intervention names for a batch of NCT IDs from ClinicalTrials.gov API v2.
    Returns a dict: {nct_id: 'DrugA; DrugB; ...'} for each trial in the batch.
    Returns empty string for trials with no intervention data.
    """
    results = {}
    url = "https://clinicaltrials.gov/api/v2/studies"
    params = {
        "filter.ids": ",".join(nct_ids),  # Comma-separated list of NCT IDs
        "fields": "NCTId,InterventionName",  # Only fetch the fields we need
        "pageSize": 100
    }
    try:
        resp = requests.get(url, params=params, timeout=30)
        if resp.status_code == 200:
            data = resp.json()
            for study in data.get("studies", []):
                nct_id = study.get("protocolSection", {}).get("identificationModule", {}).get("nctId", "")
                interventions = study.get("protocolSection", {}).get("armsInterventionsModule", {}).get("interventions", [])
                names = [i.get("name", "") for i in interventions if i.get("name")]
                results[nct_id] = "; ".join(names) if names else None
    except Exception as e:
        print(f"API error: {e}")
    return results

# ── BATCH FETCH ALL NONCOMPLIANT TRIALS ──────────────────────────────────────
# Split 92,757 NCT IDs into batches of 100 (API limit per request)
nct_list = noncompliant['nctid'].tolist()
batches  = [nct_list[i:i+100] for i in range(0, len(nct_list), 100)]
print(f"Fetching drug names for {len(nct_list):,} trials in {len(batches)} batches...")

all_interventions = {}
for idx, batch in enumerate(batches):
    result = fetch_interventions_batch(batch)
    all_interventions.update(result)
    if (idx + 1) % 100 == 0:
        print(f"  Completed {idx + 1}/{len(batches)} batches ({len(all_interventions)} found)")
    time.sleep(0.3)  # Polite rate limiting — 0.3s between requests

print(f"\nDone! Got drug names for {len(all_interventions):,} out of {len(nct_list):,} trials")


## Cell 6 — Match Trials to FAERS & Assign Danger Scores

### Matching logic
1. Clean intervention names (strip dosage, route, formulation info)
2. Try exact match against FAERS `active_ingredient` set
3. Fall back to partial string match if exact fails
4. Take the best match (highest AE count) per trial

### Danger score formula
- `danger_score_raw = log(1 + ae_count)` — log normalization reduces outlier effect
- `danger_score = (raw / max_raw) * 100` — scaled to 0–100
- Tiers: CRITICAL (>10K AEs), HIGH (1K–10K), MODERATE (100–1K), LOW (1–100), NO DATA (0)

In [ ]:
# ── MERGE DRUG NAMES INTO NONCOMPLIANT DATASET ────────────────────────────────
noncompliant = noncompliant.copy()
noncompliant['intervention_names'] = noncompliant['nctid'].map(all_interventions)

# Merge org_name and phases from trialsclean (needed for NIH lookup and grouping)
noncompliant = noncompliant.merge(
    trials[['nctid', 'orgname', 'orgclass', 'phases']],
    on='nctid', how='left'
)

# Save checkpoint — API calls are expensive, don't lose progress
noncompliant.to_csv(BASE_OLD + '/full_noncompliant_with_drugs_spark.csv', index=False)
print("Checkpoint saved after drug name fetch.")

# ── DRUG NAME CLEANING ────────────────────────────────────────────────────────
def clean_drug_names(intervention_string):
    """
    Parse and clean intervention name strings into a list of matchable drug names.
    - Removes placebo/control arms (not real drugs)
    - Strips dosage info (e.g. '500mg', 'low dose')
    - Strips route of administration (oral, IV, topical)
    - Extracts both primary name and alternative name from parentheses
    - Returns uppercase names for case-insensitive FAERS matching
    """
    if pd.isna(intervention_string):
        return []
    drugs = intervention_string.split(';')
    cleaned = []
    for drug in drugs:
        drug = drug.strip()
        # Skip control/placebo arms — these aren't real drug treatments
        skip_words = ['placebo', 'sham', 'control', 'no treatment', 'standard of care',
                      'usual care', 'waitlist', 'wait list', 'observation']
        if any(s in drug.lower() for s in skip_words):
            continue
        # Remove dosage info in parens: '(500 mg)', '(10 ml/kg)'
        drug = re.sub(r'\([\d\.\s]*(mg|ml|mcg|g|iu|units?)[\s/\w]*\)', '', drug, flags=re.IGNORECASE)
        # Extract alternative name if present: 'Ibuprofen (Advil)' → also try 'ADVIL'
        alt_match = re.search(r'\(([^)]+)\)', drug)
        alt_name = alt_match.group(1).strip() if alt_match else None
        drug = re.sub(r'\([^)]*\)', '', drug).strip()  # Remove remaining parens
        drug = re.sub(r'\b(low|high|moderate|standard)\s+dose\b', '', drug, flags=re.IGNORECASE)
        drug = re.sub(r'\b(oral|iv|injectable|topical)\s*(tablet|capsule|cream|solution)?\b', '', drug, flags=re.IGNORECASE)
        drug = drug.strip().upper()
        if drug and len(drug) > 1:
            cleaned.append(drug)
        if alt_name and len(alt_name) > 1:
            cleaned.append(alt_name.upper())
    return cleaned

# ── MATCH TRIALS TO FAERS INGREDIENTS ────────────────────────────────────────
faers_ingredients = set(ae_counts['active_ingredient'].dropna())
matches = []

for i, row in noncompliant.iterrows():
    drug_names = clean_drug_names(row['intervention_names'])
    best_ae_count = 0
    best_match    = None

    for drug in drug_names:
        if drug in faers_ingredients:
            # Exact match — most reliable
            count = ae_counts[ae_counts['active_ingredient'] == drug]['ae_count'].values[0]
            if count > best_ae_count:
                best_ae_count = count
                best_match    = drug
        else:
            # Partial match fallback — e.g. 'METFORMIN HCL' matches 'METFORMIN'
            partial = ae_counts[ae_counts['active_ingredient'].str.contains(drug, na=False, regex=False)]
            if len(partial) > 0:
                top = partial.iloc[0]
                if top['ae_count'] > best_ae_count:
                    best_ae_count = top['ae_count']
                    best_match    = top['active_ingredient']

    matches.append({'nctid': row['nctid'], 'matched_ingredient': best_match, 'ae_count': best_ae_count if best_match else 0})

    if (i + 1) % 10000 == 0:
        print(f"  Matched {i + 1:,} trials...")

matches_df = pd.DataFrame(matches)
matched = matches_df['matched_ingredient'].notna().sum()
print(f"\nMatched to FAERS: {matched:,} ({matched/len(matches_df)*100:.1f}%)")

# ── ASSIGN DANGER SCORES ──────────────────────────────────────────────────────
noncompliant = noncompliant.merge(matches_df, on='nctid', how='left')

# Log normalization: log(1 + ae_count) reduces the effect of extreme outliers
noncompliant['danger_score_raw'] = np.log1p(noncompliant['ae_count'])
max_score = noncompliant['danger_score_raw'].max()
# Scale to 0–100 for display
noncompliant['danger_score'] = ((noncompliant['danger_score_raw'] / max_score) * 100).round(1)

# Classify into 5 tiers based on raw AE count thresholds
noncompliant['danger_tier'] = pd.cut(
    noncompliant['ae_count'],
    bins=[-1, 0, 100, 1000, 10000, float('inf')],
    labels=['NO DATA', 'LOW', 'MODERATE', 'HIGH', 'CRITICAL']
)

print("\nDanger tier breakdown:")
print(noncompliant['danger_tier'].value_counts().to_string())

# Save checkpoint
noncompliant.to_csv(BASE_OLD + '/full_with_danger_scores_spark.csv', index=False)
print("Checkpoint saved after danger scoring.")


## Cell 7 — Query NIH Reporter for Public Funding at Risk
For each sponsor organization with 2+ noncompliant trials, queries the NIH Reporter API
to find total grant funding linked to that organization.

This quantifies how much **taxpayer money** is funding research where results were never reported.
Result: `$1.6B NIH funding at risk` across 6,618 organizations.

In [ ]:
def query_nih_funding(org_name):
    """
    Query NIH Reporter API v2 for total grant funding for a given organization.
    Returns (total_award_amount, grant_count) for the organization.
    Returns (0, 0) on any error to allow pipeline to continue.
    """
    url = "https://api.reporter.nih.gov/v2/projects/search"
    payload = {
        "criteria": {"org_names": [org_name]},
        "limit": 50,
        "offset": 0
    }
    try:
        resp = requests.post(url, json=payload, timeout=15)
        if resp.status_code == 200:
            data    = resp.json()
            results = data.get("results", [])
            total   = sum(r.get("award_amount", 0) or 0 for r in results)
            return total, len(results)
        return 0, 0
    except:
        return 0, 0

# ── QUERY ORGS WITH 2+ NONCOMPLIANT TRIALS ───────────────────────────────────
# Only query orgs with >= 2 trials to focus on meaningful patterns
# and avoid burning API quota on one-off trials
org_counts    = noncompliant['orgname'].value_counts()
orgs_to_query = org_counts[org_counts >= 2].index.tolist()
print(f"Querying NIH for {len(orgs_to_query):,} organizations...")

funding_records = []
for i, org in enumerate(orgs_to_query):
    amount, count = query_nih_funding(org)
    funding_records.append({
        'orgname':                  org,
        'public_dollars_at_risk':   amount,
        'funding_count':            count
    })
    if (i + 1) % 100 == 0:
        print(f"  Completed {i+1}/{len(orgs_to_query)}")
    time.sleep(0.2)  # Rate limiting — NIH Reporter allows ~5 req/sec

funding_df = pd.DataFrame(funding_records)
funded = funding_df[funding_df['public_dollars_at_risk'] > 0]
print(f"\nOrgs with NIH funding: {len(funded):,}")
print(f"Total public dollars at risk: ${funded['public_dollars_at_risk'].sum():,.0f}")


## Cell 8 — Final Merge, Deduplication & Save

### Why deduplication matters
Without deduplication, dashboard totals would be inflated:
- **AE counts**: If 500 trials all tested Aspirin, summing `ae_count` would
  count Aspirin's FAERS reports 500 times. `ae_count_deduped` only counts
  each drug's AEs once.
- **Funding**: If Pfizer has 528 noncompliant trials and $26K in NIH funding,
  summing `public_dollars_at_risk` would multiply $26K × 528.
  `funding_deduped` assigns the funding to only one trial per org.

This is how the dashboard correctly shows **$1.6B** total (not $800B).

In [ ]:
# ── MERGE FUNDING INTO ENRICHED DATASET ──────────────────────────────────────
noncompliant = noncompliant.merge(funding_df, on='orgname', how='left')
noncompliant['public_dollars_at_risk'] = noncompliant['public_dollars_at_risk'].fillna(0)
noncompliant['funding_count']          = noncompliant['funding_count'].fillna(0)

# ── BUILD FINAL risk_enrichment.csv ──────────────────────────────────────────
# Select only columns needed by MongoDB loader
risk_enrichment = noncompliant[[
    'nctid', 'orgname', 'compliancestatusv2', 'matched_ingredient',
    'ae_count', 'danger_score', 'danger_tier',
    'public_dollars_at_risk', 'funding_count'
]].copy()

# Save first version (pre-deduplication) for audit trail
risk_enrichment.to_csv(BASE_OLD + '/risk_enrichment.csv', index=False)
print("Initial risk_enrichment.csv saved.")

# ── DEDUPLICATION COLUMNS ─────────────────────────────────────────────────────
# Add ae_count_deduped and funding_deduped to avoid double-counting in dashboard totals
risk = pd.read_csv(BASE_OLD + '/risk_enrichment.csv')

# ae_count_deduped: assign AE count to only the FIRST trial matching each ingredient
# All other trials with the same ingredient get 0 — prevents summing the same AEs N times
risk['ae_count_deduped'] = 0
first_drug = risk.dropna(subset=['matched_ingredient']).drop_duplicates(subset=['matched_ingredient'])
risk.loc[first_drug.index, 'ae_count_deduped'] = first_drug['ae_count']

# funding_deduped: assign NIH funding to only the FIRST trial per sponsor org
# All other trials for that org get 0 — prevents multiplying funding by trial count
risk['funding_deduped'] = 0
first_org = risk[risk['public_dollars_at_risk'] > 0].drop_duplicates(subset=['orgname'])
risk.loc[first_org.index, 'funding_deduped'] = first_org['public_dollars_at_risk']

# Print comparison to verify deduplication is working
print(f"BEFORE deduplication (inflated):")
print(f"  Total AEs: {risk['ae_count'].sum():,.0f}")
print(f"  Total funding: ${risk['public_dollars_at_risk'].sum():,.0f}")

print(f"\nAFTER deduplication (accurate):")
print(f"  Total unique AEs: {risk['ae_count_deduped'].sum():,.0f}")
print(f"  Total unique funding: ${risk['funding_deduped'].sum():,.0f}")

# ── SAVE FINAL risk_enrichment.csv ───────────────────────────────────────────
risk.to_csv(BASE_OLD + '/risk_enrichment.csv', index=False)

print("\nFINAL risk_enrichment.csv saved!")
print(f"Shape: {risk.shape}")
print(f"Trials with AE data: {(risk['ae_count'] > 0).sum():,}")
print(f"Trials with funding data: {(risk['public_dollars_at_risk'] > 0).sum():,}")
print(f"\nDanger tier breakdown:")
print(risk['danger_tier'].value_counts().to_string())
print(f"\nTop 10 highest-risk trials:")
print(risk.nlargest(10, ['danger_score', 'public_dollars_at_risk']).to_string())
print("\nStep 2 complete. risk_enrichment.csv is ready for MongoDB loading (Step 3).")
